In [3]:
!pip install nba_api tqdm
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguegamelog
from tqdm import tqdm
import time


In [4]:
#pulling the seasons 2024-2000 from the api
headers = {
    'Host': 'stats.nba.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com'
}


seasons = []

for year in range(2000, 2025):
    seasons.append(f"{year}-{str(year+1)[-2:]}")

all_games = []

for season in tqdm(seasons):

    try:
        gamelog = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star='Regular Season',
            headers=headers
        )

        df = gamelog.get_data_frames()[0]
        df["SEASON"] = season

        all_games.append(df)

        time.sleep(2)

    except Exception as e:
        print("Failed season:", season)
        print(e)
        time.sleep(5)



100%|██████████| 25/25 [01:01<00:00,  2.44s/it]


In [5]:
print(df.shape)


(2460, 30)


In [15]:
df.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON
0,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,40,33,6,3,4,15,132,23,1,2024-25
1,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,47,17,4,1,16,22,103,-7,1,2024-25
2,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,46,22,7,8,7,22,110,7,1,2024-25
3,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,34,20,2,3,12,12,109,-23,1,2024-25
4,22024,1610612744,GSW,Golden State Warriors,0022400072,2024-10-23,GSW @ POR,W,240,48,...,57,38,13,5,18,27,140,36,1,2024-25


In [6]:
#shape of the data
games_df = pd.concat(all_games, ignore_index=True)

print(games_df.shape)


(60050, 30)


In [7]:
#total number of games in each of the seasons in the dataset
games_df["SEASON"].value_counts().sort_index()


SEASON
2000-01    2378
2001-02    2378
2002-03    2378
2003-04    2378
2004-05    2460
2005-06    2460
2006-07    2460
2007-08    2460
2008-09    2460
2009-10    2460
2010-11    2460
2011-12    1980
2012-13    2460
2013-14    2460
2014-15    2460
2015-16    2460
2016-17    2460
2017-18    2460
2018-19    2460
2019-20    2118
2020-21    2160
2021-22    2460
2022-23    2460
2023-24    2460
2024-25    2460
Name: count, dtype: int64

In [8]:
#merge home and away team stats for a game inot one single line of data 
#otherwise games will be counted twice 

games_df["HOME"] = games_df["MATCHUP"].str.contains("vs").astype(int)
home_df = games_df[games_df["HOME"] == 1]
away_df = games_df[games_df["HOME"] == 0]
merged_df = home_df.merge(
    away_df,
    on="GAME_ID",
    suffixes=("_HOME", "_AWAY")
)
print(merged_df.shape)


(30020, 61)


In [9]:
#make sure the data is ready for modeling
merged_df.duplicated(subset="GAME_ID").sum()

merged_df["HOME_WIN"] = (merged_df["WL_HOME"] == "W").astype(int)

merged_df.describe()


,TEAM_ID_HOME,MIN_HOME,FGM_HOME,FGA_HOME,FG_PCT_HOME,FG3M_HOME,FG3A_HOME,FG3_PCT_HOME,FTM_HOME,FTA_HOME,...,AST_AWAY,STL_AWAY,BLK_AWAY,TOV_AWAY,PF_AWAY,PTS_AWAY,PLUS_MINUS_AWAY,VIDEO_AVAILABLE_AWAY,HOME_AWAY,HOME_WIN
count,3.002000e+04,30020.000000,30020.000000,30020.000000,30019.000000,30020.000000,30020.000000,30019.000000,30020.000000,30020.000000,...,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.000000,30020.0,30020.000000
mean,1.610613e+09,241.731679,38.680280,83.760160,0.462651,8.449667,23.368854,0.358424,18.362858,24.072818,...,22.029714,7.579614,4.647935,14.488008,20.947801,101.417988,-2.755097,0.524184,0.0,0.585943
std,8.623176e+00,7.504740,5.545734,8.049783,0.056638,4.450568,10.145713,0.112244,6.256806,7.719660,...,5.307932,2.899661,2.451434,3.984391,4.560724,13.942128,13.586447,0.508937,0.0,0.492567
min,1.610613e+09,0.000000,0.000000,0.000000,0.247000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-73.000000,0.000000,0.0,0.000000
25%,1.610613e+09,240.000000,35.000000,78.000000,0.424000,5.000000,15.000000,0.286000,14.000000,19.000000,...,18.000000,6.000000,3.000000,12.000000,18.000000,92.000000,-11.000000,0.000000,0.0,0.000000
50%,1.610613e+09,240.000000,39.000000,83.000000,0.462000,8.000000,22.000000,0.357000,18.000000,24.000000,...,22.000000,7.000000,4.000000,14.000000,21.000000,101.000000,-4.000000,1.000000,0.0,1.000000
75%,1.610613e+09,240.000000,42.000000,89.000000,0.500000,11.000000,31.000000,0.429000,22.000000,29.000000,...,25.000000,9.000000,6.000000,17.000000,24.000000,111.000000,7.000000,1.000000,0.0,1.000000
max,1.610613e+09,340.000000,65.000000,125.000000,0.689000,29.000000,70.000000,1.000000,48.000000,64.000000,...,48.000000,22.000000,19.000000,33.000000,42.000000,176.000000,57.000000,2.000000,0.0,1.000000


In [21]:
merged_df["HOME_WIN"].value_counts()


HOME_WIN
1    17590
0    12430
Name: count, dtype: int64

In [22]:
merged_df.isna().sum()

SEASON_ID_HOME            0
TEAM_ID_HOME              0
TEAM_ABBREVIATION_HOME    0
TEAM_NAME_HOME            0
GAME_ID                   0
                         ..
PLUS_MINUS_AWAY           0
VIDEO_AVAILABLE_AWAY      0
SEASON_AWAY               0
HOME_AWAY                 0
HOME_WIN                  0
Length: 62, dtype: int64

In [24]:
merged_df = merged_df.drop_duplicates()


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

In [25]:
merged_df = merged_df.sort_values("GAME_DATE_HOME")




In [26]:
# Home team rolling stats
merged_df["HOME_PTS_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["PTS_HOME"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

merged_df["HOME_REB_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["REB_HOME"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

merged_df["HOME_AST_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["AST_HOME"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

merged_df["HOME_TOV_LAST5"] = (
    merged_df.groupby("TEAM_ID_HOME")["TOV_HOME"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)


# Away team rolling stats
merged_df["AWAY_PTS_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["PTS_AWAY"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

merged_df["AWAY_REB_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["REB_AWAY"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

merged_df["AWAY_AST_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["AST_AWAY"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

merged_df["AWAY_TOV_LAST5"] = (
    merged_df.groupby("TEAM_ID_AWAY")["TOV_AWAY"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)


In [27]:
merged_df.columns

Index(['SEASON_ID_HOME', 'TEAM_ID_HOME', 'TEAM_ABBREVIATION_HOME',
       'TEAM_NAME_HOME', 'GAME_ID', 'GAME_DATE_HOME', 'MATCHUP_HOME',
       'WL_HOME', 'MIN_HOME', 'FGM_HOME', 'FGA_HOME', 'FG_PCT_HOME',
       'FG3M_HOME', 'FG3A_HOME', 'FG3_PCT_HOME', 'FTM_HOME', 'FTA_HOME',
       'FT_PCT_HOME', 'OREB_HOME', 'DREB_HOME', 'REB_HOME', 'AST_HOME',
       'STL_HOME', 'BLK_HOME', 'TOV_HOME', 'PF_HOME', 'PTS_HOME',
       'PLUS_MINUS_HOME', 'VIDEO_AVAILABLE_HOME', 'SEASON_HOME', 'HOME_HOME',
       'SEASON_ID_AWAY', 'TEAM_ID_AWAY', 'TEAM_ABBREVIATION_AWAY',
       'TEAM_NAME_AWAY', 'GAME_DATE_AWAY', 'MATCHUP_AWAY', 'WL_AWAY',
       'MIN_AWAY', 'FGM_AWAY', 'FGA_AWAY', 'FG_PCT_AWAY', 'FG3M_AWAY',
       'FG3A_AWAY', 'FG3_PCT_AWAY', 'FTM_AWAY', 'FTA_AWAY', 'FT_PCT_AWAY',
       'OREB_AWAY', 'DREB_AWAY', 'REB_AWAY', 'AST_AWAY', 'STL_AWAY',
       'BLK_AWAY', 'TOV_AWAY', 'PF_AWAY', 'PTS_AWAY', 'PLUS_MINUS_AWAY',
       'VIDEO_AVAILABLE_AWAY', 'SEASON_AWAY', 'HOME_AWAY', 'HOME_WIN',
       'H

In [30]:
merged_df = merged_df.dropna(subset=[
    'HOME_PTS_LAST5', 'AWAY_PTS_LAST5', 'PTS_LAST5_DIFF', 'HOME_REB_LAST5',
       'HOME_AST_LAST5', 'HOME_TOV_LAST5', 'AWAY_REB_LAST5', 'AWAY_AST_LAST5',
       'AWAY_TOV_LAST5'],)


In [32]:
# Extract starting year as integer
merged_df['SEASON_START'] = merged_df['SEASON_HOME'].str[:4].astype(int)

# Split dataset
train_df = merged_df[merged_df['SEASON_START'] <= 2019]
test_df  = merged_df[merged_df['SEASON_START'] >= 2020]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (23850, 72)
Test shape: (5995, 72)


In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix

# Predictor variables
predictors = [
    'PTS_LAST5_DIFF',
    'HOME_REB_LAST5', 'AWAY_REB_LAST5',
    'HOME_AST_LAST5', 'AWAY_AST_LAST5',
    'HOME_TOV_LAST5', 'AWAY_TOV_LAST5'
]

X_train = train_df[predictors]
y_train = train_df['HOME_WIN']

X_test = test_df[predictors]
y_test = test_df['HOME_WIN']

# Train logistic regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

# Predict and evaluate
y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log Loss:", log_loss(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.5769808173477898
Log Loss: 0.6741336011943294

Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.20      0.30      2690
           1       0.58      0.88      0.70      3305

    accuracy                           0.58      5995
   macro avg       0.58      0.54      0.50      5995
weighted avg       0.58      0.58      0.52      5995


Confusion Matrix:
 [[ 537 2153]
 [ 383 2922]]
